# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Classical GNNs inherently presume homophily. Zhu et al. (2020) and Chien et al. (2021) showed that standard GCN fails on heterophilic graphs, introducing decoupled low-pass and high-pass spectral filtering. Bodnar et al. (2022) generalized graph Laplacians to **Cellular Sheaves**, associating nodes with vector stalks and edges with transport restriction maps, proving that sheaf diffusion overcomes both oversmoothing and heterophily. Sun et al. (2019) introduced **RotatE**, formalizing knowledge graph embeddings as rotations in complex Euclidean space.

### Purpose & Target Architecture
This notebook constructs from first principles:
1. Decoupled high-pass / low-pass spatial filtering for heterophilic graphs.
2. 1D orthogonal cellular sheaf Laplacian and continuous sheaf diffusion.
3. RotatE complex relational embedding score function.

# 2. Mathematical Foundations & Governing Equations

### Heterophilic Spectral Channel Decoupling
Splitting graph operations into self, low-pass, and high-pass components:
$$H^{(l+1)} = \text{ReLU}\left(H^{(l)} W_{\text{self}} + (\hat{A} H^{(l)}) W_{\text{low}} + ((I - \hat{A}) H^{(l)}) W_{\text{high}}\right)$$

### Cellular Sheaf Laplacian
For 1D signed connection sheaf $S \in \{-1, +1\}^{N \times N}$:
$$L_{\mathcal{F}} = D - S \odot A, \quad H^{(l+1)} = H^{(l)} - \Delta t \cdot L_{\mathcal{F}} H^{(l)}$$

### RotatE Complex Relation Geometry
For complex entities $h, t \in \mathbb{C}^d$ and unit phase angle relation $r = e^{i \theta}$:
$$d_r(h, t) = -\|h \circ r - t\|_2$$
where $(h \circ r)_{\text{re}} = h_{\text{re}} \cos \theta - h_{\text{im}} \sin \theta$ and $(h \circ r)_{\text{im}} = h_{\text{re}} \sin \theta + h_{\text{im}} \cos \theta$.

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Heterophilic bipartite graph: N=4 nodes, edges (0,1) and (2,3) have opposite labels
N, D = 4, 8
adj = torch.tensor([
    [0.0, 1.0, 1.0, 0.0],
    [1.0, 0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0, 1.0],
    [0.0, 1.0, 1.0, 0.0],
], dtype=torch.float32)

# Edge signs: -1 indicates disassortative connection
edge_signs = torch.tensor([
    [0.0, 1.0, -1.0, 0.0],
    [1.0, 0.0, 0.0, -1.0],
    [-1.0, 0.0, 0.0, 1.0],
    [0.0, -1.0, 1.0, 0.0],
], dtype=torch.float32)

h = torch.randn(N, D)
deg = adj.sum(dim=-1)
norm_adj = torch.diag(deg.pow(-0.5)) @ adj @ torch.diag(deg.pow(-0.5))

print(f"Setup complete: N={N} nodes, D={D} features, heterophilic signs matrix shape={edge_signs.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def heterophilic_filter(h: torch.Tensor, norm_adj: torch.Tensor, w_self: torch.Tensor, w_low: torch.Tensor, w_high: torch.Tensor) -> torch.Tensor:
    eye = torch.eye(norm_adj.shape[0])
    h_self = h @ w_self
    h_low = (norm_adj @ h) @ w_low
    h_high = ((eye - norm_adj) @ h) @ w_high
    return F.relu(h_self + h_low + h_high)

def cellular_sheaf_laplacian_1d(adj: torch.Tensor, edge_signs: torch.Tensor) -> torch.Tensor:
    deg = torch.sum(adj, dim=-1)
    return torch.diag(deg) - (edge_signs * adj)

def sheaf_diffusion_step(h: torch.Tensor, sheaf_laplacian: torch.Tensor, dt: float) -> torch.Tensor:
    return h - dt * (sheaf_laplacian @ h)

def rotate_kge_score(h_re: torch.Tensor, h_im: torch.Tensor, theta: torch.Tensor, t_re: torch.Tensor, t_im: torch.Tensor) -> torch.Tensor:
    r_re, r_im = torch.cos(theta), torch.sin(theta)
    rot_re = h_re * r_re - h_im * r_im
    rot_im = h_re * r_im + h_im * r_re
    diff_re = rot_re - t_re
    diff_im = rot_im - t_im
    return -torch.sqrt(torch.sum(diff_re ** 2 + diff_im ** 2, dim=-1, keepdim=True) + 1e-12)

w_self = torch.randn(D, D) * 0.2
w_low = torch.randn(D, D) * 0.2
w_high = torch.randn(D, D) * 0.2

h_out = heterophilic_filter(h, norm_adj, w_self, w_low, w_high)
sheaf_laplacian = cellular_sheaf_laplacian_1d(adj, edge_signs)
h_diff = sheaf_diffusion_step(h, sheaf_laplacian, dt=0.1)

print(f"Heterophilic filter complete: output shape = {h_out.shape}")
print(f"Sheaf diffusion complete: output shape = {h_diff.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: High-pass + Low-pass reconstruction: A_low + A_high == I
eye = torch.eye(N)
high_pass_adj = eye - norm_adj
assert torch.allclose(norm_adj + high_pass_adj, eye, atol=1e-6), "Spectral filter completeness must sum to Identity"

# Invariant 2: Sheaf Laplacian self-adjointness (symmetry)
assert torch.allclose(sheaf_laplacian, sheaf_laplacian.T, atol=1e-6), "Sheaf Laplacian must be symmetric for undirected graphs"

# Invariant 3: RotatE identity rotation when theta = 0
b_test, d_test = 2, 4
h_re_test = torch.randn(b_test, d_test)
h_im_test = torch.randn(b_test, d_test)
zero_theta = torch.zeros(b_test, d_test)
id_score = rotate_kge_score(h_re_test, h_im_test, zero_theta, h_re_test, h_im_test)
assert torch.allclose(id_score, torch.zeros(b_test, 1), atol=1e-5), "Distance of rotated entity to itself at theta=0 must be zero"
print("All heterophilic, sheaf, and KGE invariants passed successfully!")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark standard GCN smoothing vs Sheaf diffusion
start = time.perf_counter()
standard_laplacian = torch.diag(deg) - adj
for _ in range(500):
    _ = sheaf_diffusion_step(h, standard_laplacian, dt=0.05)
std_diff_time = (time.perf_counter() - start) / 500.0

start = time.perf_counter()
for _ in range(500):
    _ = sheaf_diffusion_step(h, sheaf_laplacian, dt=0.05)
sheaf_diff_time = (time.perf_counter() - start) / 500.0

print(f"Standard Laplacian diffusion latency: {std_diff_time*1e6:.1f} µs")
print(f"Cellular Sheaf diffusion latency: {sheaf_diff_time*1e6:.1f} µs")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Classical GCN smoothing on an adversarial heterophilic graph
# Nodes 0 and 1 have opposite sign labels (+1 vs -1)
x_hetero = torch.tensor([[1.0], [-1.0], [1.0], [-1.0]])

# Standard GCN averaging washes out the opposing classes
gcn_smoothed = norm_adj @ x_hetero
gcn_separation = torch.abs(gcn_smoothed[0] - gcn_smoothed[1]).item()

# Sheaf diffusion with negative sign preserves antipodal separation
sheaf_smoothed = sheaf_diffusion_step(x_hetero, sheaf_laplacian, dt=0.2)
sheaf_separation = torch.abs(sheaf_smoothed[0] - sheaf_smoothed[1]).item()

print(f"Classical GCN class separation after smoothing: {gcn_separation:.4f} (Severe feature collapse)")
print(f"Sheaf Laplacian class separation after diffusion: {sheaf_separation:.4f} (Preserves antipodal signal)")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Insights
1. **Beyond Homophily via Sheaves**: Standard graph Laplacians can only model diffusion towards a consensus constant vector. Cellular sheaves provide non-trivial parallel transport maps, enabling antipodal diffusion.
2. **High-Pass Spectral Channels**: Decoupling high-pass $I - \hat{A}$ from low-pass $\hat{A}$ preserves contrasting features in disassortative relational structures.
3. **RotatE Relational Expressivity**: Complex rotations model symmetry, antisymmetry, and inversion within an abelian group structure without parameter explosion.